In [ ]:
import os
import pandas as pd
import numpy as np
os.chdir("C://kurssit/Dippatyo/tutkielma/data_analyysit/soccer_hmm_project")
from src.dataset import make_data_dict
from src.dataset import concatenate_dfs
import matplotlib.pyplot as plt
from scipy.stats import vonmises

In [ ]:
print(os.getcwd())

In [ ]:
non_filt = make_data_dict(directory="data/raw/Catapult_RAW", preprocess_data=True)

In [ ]:
df = non_filt[16]
df[~df.isna()]["segment_id"].value_counts()

In [ ]:
df[~df["ar_10hz"].isna()]["segment_id"].value_counts()

In [ ]:
df[df["segment_id"] == 25]["ar_10hz"].plot()

In [ ]:
df["acc_x"][1000:1100].plot()

In [ ]:
df["acc_x_dyn"][1000:1100].plot()

In [ ]:
df["ar_10hz"][1000:1100].plot()

In [ ]:
df[df["acc_x"].isna()]

In [ ]:
df["acc_x"][1000:1100].plot()
df["acc_x_filt"][1000:1100].plot()

In [ ]:
df["acc_y"][1000:1100].plot()
df["acc_y_filt"][1000:1100].plot()

In [ ]:
df["acc_z"][1000:1100].plot()
df["acc_z_filt"][1000:1100].plot()

In [ ]:
df["pl_filt"] = np.sqrt(df["acc_x_filt"].diff()**2 + df["acc_x_filt"].diff()**2 + df["acc_x_filt"].diff()**2)
df["pl"] = np.sqrt(df["acc_x"].diff()**2 + df["acc_x"].diff()**2 + df["acc_x"].diff()**2)
df["vec_magn"] = np.sqrt(df["acc_x_filt"]**2 + df["acc_y_filt"]**2 + df["acc_z_filt"]**2)
df["ar"] = np.abs(df["vec_magn"].diff())


In [ ]:
df["vec_magn"].describe().round(3)

In [ ]:
df[["pl", "pl_filt"]].describe().round(3)

In [ ]:
df["ar"].plot()

In [ ]:
df["pl_filt"][1200:1300].plot()

In [ ]:
df["speed"].plot()

In [ ]:
df["pl_filt"].plot()

In [ ]:
plt.scatter("pl_filt", "ar", data=df)

In [ ]:
df = pd.read_csv("data/processed/soccerdata_processed.csv", sep=",")
(df["turn_angle"] < -np.pi).any()

In [ ]:
vonmises.cdf(-np.pi, kappa=3, loc=0.1)

In [ ]:
df[df["track_id"] == 3]["playerload"].plot()

In [ ]:
plt.plot("speed", data=df[(df["track_id"] == 3) & (df["segment_id"] == 42)])

In [ ]:
df[(df["track_id"] == 3) & (df["segment_id"] == 30)]

In [ ]:
def preprocess_moi(df, track_id):
    col_renames = dict({
        "Excel Timestamp": "timestamp",
        "Latitude": "lat",
        "Longitude": "lon",
        "Speed": "speed",
        "Accel X": "acc_x",
        "Accel Y": "acc_y",
        "Accel Z": "acc_z"
    })

    df.columns = df.columns.str.strip()
    #df.drop(columns=["Heart Rate"], axis=1, inplace=True)
    df.rename(columns=col_renames, inplace=True)

    #parse time, sort index
    df["timestamp"] = pd.to_datetime('1899-12-30') + pd.to_timedelta(df["timestamp"], unit="D")
    df = df.sort_values("timestamp")
    df = df.set_index("timestamp")
    df.index = df.index.round("100ms")

    #convert lat/lon to local meters
    R = 6371000  # Earth radius (m)

    lat_rad = np.deg2rad(df["lat"])
    lon_rad = np.deg2rad(df["lon"])

    lat0 = lat_rad.iloc[0]
    lon0 = lon_rad.iloc[0]
    
    #local equirectangular projection
    df["x"] = R * (lon_rad - lon0) * np.cos(lat0)
    df["y"] = R * (lat_rad - lat0)

    fs = 10
    dt_target = 1 / fs

    #reindex to exact 10 Hz
    full_index = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq=f"{int(dt_target * 1000)}ms"
    )

    df = df.reindex(full_index)
    df.reset_index(names="timestamp", inplace=True)

    # interpolate only short gaps (limit=1 means max 1 consecutive NaN)
    df[["acc_x", "acc_y", "acc_z", "speed", "x", "y"]] = (
        df[["acc_x", "acc_y", "acc_z", "speed", "x", "y"]]
          .interpolate(method="linear", limit=1)
    )


    #calculate instantaneous player load
    df["pl_10hz"] = np.sqrt(
        np.diff(df["acc_x"], prepend=df["acc_x"].iloc[0])**2 +
        np.diff(df["acc_y"], prepend=df["acc_y"].iloc[0])**2 +
        np.diff(df["acc_z"], prepend=df["acc_z"].iloc[0])**2
    )/10

    df["second"] = df["timestamp"].dt.floor("s")

    pl_1hz = (
        df.groupby("second")
            .agg(pl_raw=("pl_10hz", "sum"),
                 n_valid=("pl_10hz", "count"))
    )

    pl_1hz["coverage"] = pl_1hz["n_valid"] / 10
    pl_1hz["pl_scaled"] = pl_1hz["pl_raw"] * (10 / pl_1hz["n_valid"])
    pl_1hz.loc[pl_1hz["n_valid"] < 8, "pl_scaled"] = np.nan

    #Speed to 1 Hz
    speed_1hz = (
        df.groupby("second")
            .agg(speed=("speed", "mean"),
                 n_valid=("speed", "count"))
    )

    speed_1hz.loc[speed_1hz["n_valid"] < 8, "speed"] = np.nan

    loc_1hz = (
        df.groupby("second")
            .agg(x=("x", "last"),
                 y=("y", "last"),
                 n_valid=("x", "count"))
    )

    loc_1hz.loc[loc_1hz["n_valid"] < 8, ["x", "y"]] = np.nan

    df_1hz = pd.DataFrame({#"timestamp": df["timestamp"],
                          "x": loc_1hz["x"],
                          "y": loc_1hz["y"],
                          "speed": speed_1hz["speed"],
                          "playerload": pl_1hz["pl_scaled"]})
    
    df_1hz["track_id"] = track_id

    return df_1hz

In [ ]:
data_dict = make_data_dict(directory="data/raw/Catapult_RAW", preprocess_data=True)

In [ ]:
def find_nan_segments(series: pd.Series):
    mask = series.isna()
    groups = (mask != mask.shift()).cumsum()

    return (
        series[mask]
        .groupby(groups)
        .agg(
            start=lambda x: x.index[0],
            end=lambda x: x.index[-1],
            length="size"
        )
        .reset_index(drop=True)
    )


In [ ]:
df=data_dict[14].dropna()
df.groupby("segment_id").count()


In [ ]:
data_dict[14]["playerload"].plot()

In [ ]:
import matplotlib.pyplot as plt
df=data_dict[10].dropna()
plt.scatter(x="x", y="y", data=df)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["animation.html"] = "jshtml"

track = df[df["segment_id"] == 2].dropna(subset=["x", "y"]).copy()
track = track.sort_index()

x = track["x"].values
y = track["y"].values

fig, ax = plt.subplots()
ax.set_aspect("equal")

ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())

point, = ax.plot([], [], "ro")
trail, = ax.plot([], [], "b-", alpha=0.5)

def init():
    point.set_data([[], []])
    trail.set_data([], [])
    return point, trail

def update(frame):
    point.set_data([x[frame]], [y[frame]])
    trail.set_data(x[:frame], y[:frame])
    return point, trail

ani = FuncAnimation(
    fig,
    update,
    frames=len(track),
    init_func=init,
    interval=1000,  # 1 Hz playback
    blit=True
)

HTML(ani.to_jshtml())

In [ ]:
df["speed"].plot()

In [ ]:
nans = find_nan_segments(data_dict[4]["turn_angle"])
nans["length"].describe()

In [ ]:
nans

In [ ]:
data_dict[4]["playerload"].plot()

In [ ]:
df=data_dict[4]

In [ ]:
df.loc[df.isna().any(axis=1), ["x", "y", "speed", "playerload", "turn_angle"]] = np.nan

In [ ]:
df.isna().sum()

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(x="x", y="y", data=data_dict[9][1000:1100])

In [ ]:
nans = find_nan_segments(df["turn_angle"])
nans["length"].describe()

In [ ]:
nans

In [ ]:
for key in data_dict.keys():
    data_dict[key] = restore_full_index(data_dict[key])
    print(f"Track: {key}")
    print(f"Fraction of missing values: {data_dict[key]["speed"].isna().sum()/data_dict[key].shape[0]}")

    nan_segments = find_nan_segments(data_dict[key]["speed"])

    print(nan_segments["length"].describe())
    print(f"Number of missing segments with length over 0,5 s: {np.sum((nan_segments["length"] > 5))}")
    #print(data_dict[key]["turn_angle"].isna().sum())
    print("================================================")
    

In [ ]:
nan_segments[nan_segments["length"] > 5]

In [ ]:
df = pd.DataFrame()

for key in data_dict.keys():
    df = pd.concat([df, data_dict[key]])

In [ ]:
df.isna().sum()

In [ ]:
pd.set_option('display.max_rows', 500)

In [ ]:
df

In [ ]:
df1 = concatenate_dfs(data_dict)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
data_dict["Game_1_hyökkääjä_NN_Catapult-Export-1770541853306"].describe()

In [ ]:
for key in data_dict.keys():

    df=data_dict[key]

    df["acc_mag"] = np.sqrt(
        df["acc_x"]**2 +
        df["acc_y"]**2 +
        df["acc_z"]**2
    )

    print(df["acc_mag"].mean())

In [ ]:
np.deg2rad(np.nan)